# TP 2

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 500)
import numpy as np
import matplotlib.pyplot as plt
import seaborn.objects as so

#### Consigna 1 

In [ ]:
tabla_nutricional = pd.read_csv('./data/tabla_nutricional.csv', sep = ';')

In [ ]:
tabla_nutricional = tabla_nutricional.fillna(0)

#Paso de mg a gr.
tabla_nutricional['Na (mg)'] = tabla_nutricional['Na (mg)']/1000

tabla_nutricional['Fe (mg)'] = tabla_nutricional['Fe (mg)']/1000

tabla_nutricional['Ca (mg)'] = tabla_nutricional['Ca (mg)']/1000

#Renombro las columnas de mg a gr
tabla_nutricional = tabla_nutricional.rename(columns = {'Na (mg)' : 'Na (gr)', 'Fe (mg)' : 'Fe (gr)', 'Ca (mg)' : 'Ca (gr)'})

In [ ]:
tabla_nutricional.head()

#### Consigna 2

In [ ]:
#Tomo los gramos de la suma de las Grasas, HC y proteinas como el 100%.
gramos_totales = tabla_nutricional['Grasas (gr)'].sum() +  tabla_nutricional['HC (gr)'].sum() +  tabla_nutricional['Proteinas (gr)'].sum()

#Hardcodeo la los margenes de ingesta de la oms, pasando los porcentajes a gramos tomando como 100% a gramos_totales.
margenes_oms = {
    'Proteinas (gr)': (gramos_totales*0.10 , gramos_totales*0.15),
    'HC (gr)' : (gramos_totales*0.55, gramos_totales*0.75),
    'Grasas (gr)' : (gramos_totales*0.15, gramos_totales*0.30),
    'Na (gr)' : (0, 2),
    'Fibra (gr)' : (25, 9999), 
    'Frutas y Verduras' : (400, 9999) #Pongo 9999 porque en la tesis no especifica un maximo.
}

frutas_verduras = ['Acelga', 'Zanahoria', 'Tomate', 'Lechuga', 'Cebolla', 
                   'Zapallo', 'Manzana', 'Naranja', 'Mandarina', 'pera', 
                   'Banana','Papa', 'Batata', 'Arvejas en lata', 
                   'Tomate envasado']

# Función para aplicar a cada fila la cantidad de gramos que hay en el caso de ser una fruta o verdura.
def calc_frutas_verduras(df):
    if df['Alimento'] in frutas_verduras:
        return df['Cantidad (gr/ml)']
    else:
        return 0

#Recibe un dataframe y chequea si cumple o no las metas de la OMS.
def dieta_saludable(df):
    # Crear la nueva columna
    A = df.copy()
    # Agrego una columna con la cantidad de frutas y verduras
    A['Frutas y Verduras'] = A.apply(calc_frutas_verduras, axis=1)
    res = True
    for nutriente in margenes_oms:
        if ((margenes_oms[nutriente][0] <= A[nutriente].sum() < margenes_oms[nutriente][1]) == False):
          return False
    return res

In [ ]:
print('Pasa las metas de la OMS?', dieta_saludable(tabla_nutricional))

La dieta de la canasta basica no pasa las metas de la OMS. Por lo que se la podria considerar una dieta poco saludable.

#### Consigna 3

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# #Pasamos el dataframe a matriz de numpy
# X = tabla_nutricional.drop('Alimento', axis=1).to_numpy()
# for i in range(0, len(X)):
#     X[i,:] /= X[i, 0]

# X = X[:, 1:]

# #Escalamos la matriz con StandardScaler 
# X_star = StandardScaler().set_output(transform="pandas").fit_transform(X)

cant = tabla_nutricional['Cantidad (gr/ml)']
alim = tabla_nutricional['Alimento']

X_star = tabla_nutricional.drop(['Alimento','Cantidad (gr/ml)'], axis=1)
X_star = X_star.to_numpy()

for i in range(len(X_star)):
    for j in range(len(X_star[0])):
         X_star[i][j]/=cant[i]

#Calculamos traspuesta, dimensión y matriz sigma
X_star_t = np.transpose(X_star)
N = len(X_star)
Sigma = (X_star_t @ X_star) / N

gamma, U = np.linalg.eigh(Sigma)  # Usamos eigh porque A es simétrica. Podemos usar eig también

gamma = np.flip(gamma)
U = np.flip(U, 1)   # U tiene los autovectores de Sigma como columnas

Z = X_star @ U
Z = pd.DataFrame(Z)
Z.columns = ["Z" + str(i) for i in range(1,12)]  # Renombramos las columnas
Z.insert(0, 'Alimento', tabla_nutricional['Alimento'])

In [ ]:
(
    so.Plot()
    .add(so.Dot(),data = Z, x = "Z1", y = "Z2")
    .layout(size=(16, 9))    
)

In [ ]:
#Hacemos esta figura solo para diferenciar cada alimento.

plt.figure(figsize=(20,20))
plt.scatter(Z['Z1'], Z['Z2'], alpha=0.5)


plt.xlabel('X')
plt.ylabel('Y')

#Agrgego índices
for i in range(len(Z)):
    plt.text(Z['Z1'][i], Z['Z2'][i], Z.index[i])

plt.show()

In [ ]:
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(projection='3d')

ax.scatter(Z['Z1'], Z['Z2'], Z['Z3'])

for i in range(len(Z)):
    ax.text(Z['Z1'][i], Z['Z2'][i], Z['Z3'][i], Z.index[i])

ax.axes.set_xlim3d(left=-3, right=3) 
ax.axes.set_ylim3d(bottom=-3, top=3) 
ax.axes.set_zlim3d(bottom=-3, top=3) 

ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
ax.set_zlabel('PCA 3')

plt.show()

Identificamos cuatro clusters. Tres por debajo y uno en la parte superior. El superior está conformado mayormente por productos grasos como el aceite, o la manteca. El inferior izquierdo está conformado por galletitas y dulces. El inferior medio por harinas. Y el inferior derecho por frutas, verduras, carnes, entre otros.

#### Consigna 4

In [ ]:
consumidores_libres = pd.read_csv('./data/consumidores_libres.csv', sep = ';')
consumidores_libres.head()

In [ ]:
alm = [('Aceite girasol','ACEITE COCINERO GIRASOL'), ('Arroz','ARROZ GRANO FINO'), ('Azucar','AZUCAR LEDESMA'),('Fideos secos','FIDEOS GUISEROS'), ('Harina trigo','HARINA DE TRIGO'), ('Huevo','HUEVOS COLOR DOCENA'),('Pan Frances','PAN FRESCO'), ('Leche fluida entera','LECHE c/VIT.FORT'), ('Yerba','YERBA TARAGUI'), ('Zanahoria','ZANAHORIAS'), ('Tomate','TOMATE PERITA'), ('Cebolla','CEBOLLA'), ('Papa','PAPA NEGRA'), ('Acelga','ACELGA PAQUETE'),('Naranja','NARANJA'), ('Manzana','MANZANA'), ('Bola de Lomo','BOLA DE LOMO'), ('Asado','ASADO'), ('Paleta ','PALETA'), ('Carne picada','CARNE PICADA COMUN')]

tabla_nutricional_cl = tabla_nutricional[tabla_nutricional['Alimento'].isin(list(zip(*alm))[0])]
tabla_nutricional_cl = tabla_nutricional_cl.reset_index(drop = True)
tabla_nutricional_cl.head()

In [ ]:
# Estandarisamos
cant_cl = tabla_nutricional_cl['Cantidad (gr/ml)']
alim_cl = tabla_nutricional_cl['Alimento']

X_star_cl = tabla_nutricional_cl.drop(['Alimento','Cantidad (gr/ml)'], axis=1)
X_star_cl = X_star_cl.to_numpy()

for i in range(len(X_star_cl)):
    for j in range(len(X_star_cl[0])):
         X_star_cl[i][j]/=cant_cl[i]

#Aplicamos proyecciones con el PCA anterior
Z_cl = X_star_cl @ U
Z_cl = pd.DataFrame(Z_cl)
Z_cl.columns = ["Z" + str(i) for i in range(1,12)]  # Renombramos las columnas
Z_cl.insert(0, 'Alimento', tabla_nutricional_cl['Alimento'])

In [ ]:
(
    so.Plot()
    .add(so.Dot(),data = Z_cl, x = "Z1", y = "Z2")
    .layout(size=(16, 9))    
)

Acá vemos los datos de consumidores libres proyectados en el espacio de generadores del ACP.

#### Consigna 5

In [ ]:
#Genero una tabla que muestra el costo por gramo de cada nutriente (proteínas, grasas y carbohidratos) para cada alimento.

nut = ['Proteinas (gr)', 'Grasas (gr)', 'HC (gr)']
alm = alm
dias = ['31/12/2023', '31/1/2024', '29/2/2024', '31/3/2024', '30/4/2024']

#Iicializo el dataframe vacio.
precio_ev = pd.DataFrame({'Alimento': [''] * (20 * 5), 'Proteinas (gr)': np.zeros(20*5), 'Grasas (gr)': np.zeros(20*5), 'HC (gr)': np.zeros(20*5), 'Dia': [''] * (20 * 5)})


i = 0
for dia in dias:
    for alimento in alm:
        tabla = tabla_nutricional[tabla_nutricional['Alimento'] == alimento[0]]
        cons = consumidores_libres[consumidores_libres['PRODUCTOS'] == alimento[1]]
        preciog = cons[dia].values[0] / cons['Cantidad'].values[0]

        for nutriente in nut:
            if tabla[nutriente].values[0] != 0:  # Verificar que no sea cero
                precio_nutriente = tabla['Cantidad (gr/ml)'].values[0] / tabla[nutriente].values[0]
            else:
                precio_nutriente = np.nan  # Asignar np.nan si es cero
            precio_nutriente_g = preciog * precio_nutriente

            precio_ev.loc[i,'Alimento'] = alimento[1]
            precio_ev.loc[i,nutriente] = precio_nutriente_g
            precio_ev.loc[i,'Dia']= dia

        i += 1

precio_ev = precio_ev.replace([np.inf, -np.inf], np.nan)

In [ ]:
#Esta funcion recibe un dataframe te devuelve los coeficientes haciendo de minimos cuadrados.
def cuadrados_minimos(tabla):
    
    #Creo una columna que tenga el numero de cada dia (los dias fueron enumerados del 1 al 5)
    tabla['Dia_Num'] = tabla['Dia'] 
    tabla = tabla.replace({'Dia_Num' : '31/12/2023'}, 1)
    tabla = tabla.replace({'Dia_Num' : '31/1/2024'}, 2)
    tabla = tabla.replace({'Dia_Num' : '29/2/2024'}, 3)
    tabla = tabla.replace({'Dia_Num' : '31/3/2024'}, 4)
    tabla = tabla.replace({'Dia_Num' : '30/4/2024'}, 5)


    # Construir la matriz A
    A = np.zeros((len(tabla), 2))
    A[:, 0] = tabla['Dia_Num']
    A[:, 1] = np.ones(len(tabla))

    # Construir la matriz M y el vector N
    M = A.T @ A
    N = A.T @ tabla.iloc[:,1]

    # Resolver el sistema de ecuaciones
    c0, c1 = np.linalg.solve(M, N)
    
    return c0, c1

In [ ]:
import seaborn.objects as so

###### Proteina

In [ ]:
prote = precio_ev.drop('Grasas (gr)', axis = 1).drop('HC (gr)', axis = 1)
prote = prote.dropna()
coeficientes_prote = cuadrados_minimos(prote)
coeficientes_prote

In [ ]:
(
    so.Plot(
         x = precio_ev['Dia'],
         y = precio_ev['Proteinas (gr)']

    )
    .add(so.Dot(color = 'black', alpha = '0.3'))
    #.add(so.Dot(color = 'red', pointsize= 7), so.Agg(), label ='Promedio por dia')

    .add(so.Line(color= 'red'), x=['31/12/2023','30/4/2024'], y=[coeficientes_prote[1] + coeficientes_prote[0], coeficientes_prote[1] + coeficientes_prote[0]*5])    

    .label(title = 'Proteina', y = 'Precio ($)')
)

###### Grasas

In [ ]:
#Calculo los coeficientes con cuadrados minimos para analizar el aumento de las grasas

grasas = precio_ev.drop('Proteinas (gr)', axis = 1).drop('HC (gr)', axis = 1)
grasas = grasas.dropna()
coeficientes_grasas = cuadrados_minimos(grasas)
coeficientes_grasas

In [ ]:
(
    so.Plot(
         x = precio_ev['Dia'],
         y = precio_ev['Grasas (gr)']

    )
    .add(so.Dot(color = 'black', alpha = '0.3'))
    #.add(so.Dot(color = 'blue', pointsize= 7), so.Agg(), label = 'Promedio por dia')

    .add(so.Line(color= 'blue'), x=['31/12/2023','30/4/2024'], y=[coeficientes_grasas[1]+ coeficientes_grasas[0] ,coeficientes_grasas[1] + coeficientes_grasas[0]*5])    


    .label(title = 'Grasas', y = 'Precio ($)')
)

###### Carbohidratos

In [ ]:
#Calculo los coeficientes con cuadrados minimos para analizar el aumento de los carbohidratos

carbohidratos = precio_ev.drop('Proteinas (gr)', axis = 1).drop('Grasas (gr)', axis = 1)
carbohidratos = carbohidratos.dropna()
coeficientes_HC = cuadrados_minimos(carbohidratos)
coeficientes_HC

In [ ]:
(
    so.Plot(
         x = precio_ev['Dia'],
         y = precio_ev['HC (gr)']

    )
    .add(so.Dot(color = 'black', alpha = '0.3'))
    #.add(so.Dot(color = 'green', pointsize= 7), so.Agg(), label = 'Promedio por dia')

    .add(so.Line(color= 'green'), x=['31/12/2023','30/4/2024'], y=[coeficientes_HC[1] + coeficientes_HC[0], coeficientes_HC[1] + coeficientes_HC[0]*5])
    
    .label(title = 'Carbohidratos', y = 'Precio ($)')
)

Despues de ver estos graficos y analizar la evolucion del precio en estos ultimos meses, se ve claramente el aumento que hubo en los alimentos. En el siguiente punto vamos a analizar el como se vio afectado cada rubro (Carnes, Almacen, Frutas y Verduras).

##### Consigna 6

In [ ]:
tabla_de_precios = consumidores_libres

#Reorganizo los datos para que me queden mas faciles de usar.
tabla_de_precios = pd.melt(tabla_de_precios, id_vars=['PRODUCTOS'], value_vars=['31/12/2023', '31/1/2024', '29/2/2024', '31/3/2024', '30/4/2024'], var_name='Fecha', value_name='Precio')

In [ ]:
#Creo una columna que me diga a que rubro pertenece cada alimento (Carnes, Almacen o Fruta y Verdura).

tabla_de_precios['Rubro'] = np.zeros(len(tabla_de_precios))
carnes =  ['BOLA DE LOMO', 'ASADO', 'PALETA', 'CARNE PICADA COMUN']
almacenes = ['ACEITE COCINERO GIRASOL', 'ARROZ GRANO FINO', 'AZUCAR LEDESMA', 'FIDEOS GUISEROS', 'HARINA DE TRIGO', 'HUEVOS COLOR DOCENA', 'PAN FRESCO', 'LECHE c/VIT.FORT', 'YERBA TARAGUI']
frutayverdura = ['ZANAHORIAS', 'BERENJENAS', 'TOMATE PERITA', 'CEBOLLA', 'PAPA NEGRA', 'ACELGA PAQUETE', 'NARANJA', 'MANZANA']

# Crear un diccionario de mapeo
rubro_map = {producto: 'Carne' for producto in carnes}
rubro_map.update({producto: 'Almacen' for producto in almacenes})
rubro_map.update({producto: 'Fruta y Verdura' for producto in frutayverdura})

# Usar el método map para asignar los valores a la columna 'Rubro'
tabla_de_precios['Rubro'] = tabla_de_precios['PRODUCTOS'].map(rubro_map)


In [ ]:
# Separo la tabla de precios segun cada rubro.

# Tabla para a carne
df_carnes = tabla_de_precios[tabla_de_precios['Rubro'] == 'Carne']
df_carnes = df_carnes.rename(columns = {'Fecha' : 'Dia'}) # Cambio el nombre de Fecha a Dia para que coincida con precio_ev.
df_carnes = df_carnes[['PRODUCTOS', 'Precio', 'Dia', 'Rubro']] # Cambio el orden de las columnas.

# Tabla para las frutas y verduras
df_fyv = tabla_de_precios[tabla_de_precios['Rubro'] == 'Fruta y Verdura']
df_fyv = df_fyv.rename(columns = {'Fecha' : 'Dia'}) # Cambio el nombre de Fecha a Dia para que coincida con precio_ev.
df_fyv = df_fyv[['PRODUCTOS', 'Precio', 'Dia', 'Rubro']] # Cambio el orden de las columnas.

# Tabla para el almacen
df_almacen = tabla_de_precios[tabla_de_precios['Rubro'] == 'Almacen']
df_almacen =  df_almacen.rename(columns = {'Fecha' : 'Dia'}) # Cambio el nombre de Fecha a Dia para que coincida con precio_ev.
df_almacen = df_almacen[['PRODUCTOS', 'Precio', 'Dia', 'Rubro']] # Cambio el orden de las columnas.

In [ ]:
#Calculo los coeficientes de cada rubro con cuadrados minimos.

coeficientes_carne = cuadrados_minimos(df_carnes)
coeficientes_almacen = cuadrados_minimos(df_almacen)
coeficientes_frutasyverduras = cuadrados_minimos(df_fyv)

In [ ]:
(
    so.Plot(
        x = tabla_de_precios['Fecha'],
        y = tabla_de_precios['Precio'],
        color = tabla_de_precios['Rubro']
    )
    .add(so.Dot())
        
    .add(so.Dot(edgecolor = 'black', pointsize = 12), so.Agg())

    .add(so.Line(color= 'blue'), x=['31/12/2023','30/4/2024'], y=[coeficientes_almacen[1] + coeficientes_almacen[0],coeficientes_almacen[1] + coeficientes_almacen[0]*5], legend= False)

    .add(so.Line(color= 'green'), x=['31/12/2023','30/4/2024'], y=[coeficientes_carne[1] + coeficientes_carne[0], coeficientes_carne[1] + coeficientes_carne[0]*5], legend= False)

    .add(so.Line(color= 'orange'), x=['31/12/2023','30/4/2024'], y=[coeficientes_frutasyverduras[1] + coeficientes_frutasyverduras[0], coeficientes_frutasyverduras[1] + coeficientes_frutasyverduras[0]*5], legend= False)
    
    .label(title = 'Comparacion entre el aumento de la carne y el de los otros rubros')
)

In [ ]:
#Calculo el aumento de la carne

#Armo una tabla con los promedios del precio de la carne en cada fecha.

carnes = tabla_de_precios[tabla_de_precios['Rubro'] == 'Carne']
prom_carnes = carnes.drop('PRODUCTOS', axis = 1).drop('Rubro', axis = 1).groupby(['Fecha']).mean()
prom_carnes.index = pd.to_datetime(prom_carnes.index, dayfirst=True)
prom_carnes_ord = prom_carnes.sort_index()
prom_carnes_ord.index =  range(1, 6)
#Armo una tabla con los promedios del precio de los alimentos de almacen en cada fecha.

almacen = tabla_de_precios[tabla_de_precios['Rubro'] == 'Almacen']
prom_almacen = almacen.drop('PRODUCTOS', axis = 1).drop('Rubro', axis = 1).groupby(['Fecha']).mean()
prom_almacen.index = pd.to_datetime(prom_almacen.index, dayfirst=True)
prom_almacen_ord = prom_almacen.sort_index()
prom_almacen_ord.index =  range(1, 6)

#Armo una tabla con los promedios del precio de las frutas y las verduras en cada fecha.

frutasyverduras = tabla_de_precios[tabla_de_precios['Rubro'] == 'Fruta y Verdura']
prom_frutasyverduras = frutasyverduras.drop('PRODUCTOS', axis = 1).drop('Rubro', axis = 1).groupby(['Fecha']).mean()
prom_frutasyverduras.index = pd.to_datetime(prom_frutasyverduras.index, dayfirst=True)
prom_frutasyverduras_ord = prom_frutasyverduras.sort_index()
prom_frutasyverduras_ord.index =  range(1,6)

aumento_carne = prom_carnes_ord.iloc[4,:]/prom_carnes_ord.iloc[0,:]*100 - 100
print('El aumento de la carne fue del:', aumento_carne.item(),'%')

#Calculo el aumento de almacen

aumento_almacen= prom_almacen_ord.iloc[4,:]/prom_almacen_ord.iloc[0,:]*100 -100
print('El aumento de almacen fue del:', aumento_almacen.item(),'%')

#Calculo el aumento de las frutas y verduras

aumento_frutasyverduras = prom_frutasyverduras_ord.iloc[4,:]/prom_frutasyverduras_ord.iloc[0,:]*100 -100
print('El aumento de frutas y verduras fue del:', aumento_frutasyverduras.item(),'%')

In [ ]:
alim_carne = ['Asado', 'Paleta', 'Hueso c/carne', 'Carne picada','Bola de Lomo', 'Higado', 'pollo', 'pescado','Mortadela','Paleta cocida']

#Gracias a este aumento el consumo de carne bajo un 18.5 %, vamos a ver como afecta esto a la ingesta individual.

#Creo una nueva tabla (tn) que tenga un 18.5 % menos de carne que la de tabla_nutricional.

tabla_actualizada = tabla_nutricional.loc[tabla_nutricional['Alimento'].isin(alim_carne)]
tn = tabla_nutricional.drop(tabla_nutricional[tabla_nutricional['Alimento'].isin(alim_carne)].index)

alimentos = tabla_actualizada['Alimento']
tabla_actualizada = tabla_actualizada.drop(['Alimento'], axis=1).mul(0.815, axis= 0)

tabla_actualizada.insert(0, "Alimento", alimentos)

tn = tn.merge(tabla_actualizada, how = 'outer')

print('Con la tabla actualizada la dieta cumple los estandares de la OMS?',dieta_saludable(tn))

Con el 18.5% menos de carne la dieta sigue sin cumplir las metas de la OMS.

#### Consigna 7

In [ ]:
#Calculo la diferencia que hay en valor nutricional entre la tabla_nutricional y la tabla actualizada con un 18.5 % menos de carne.

dif_prote = tabla_nutricional['Proteinas (gr)'].sum() - tn['Proteinas (gr)'].sum()
dif_hc = tabla_nutricional['HC (gr)'].sum() - tn['HC (gr)'].sum()
dif_grasa = tabla_nutricional['Grasas (gr)'].sum() - tn['Grasas (gr)'].sum()
dif_fibra = tabla_nutricional['Fibra (gr)'].sum() - tn['Fibra (gr)'].sum()
dif_na = tabla_nutricional['Na (gr)'].sum() - tn['Na (gr)'].sum()

dif_nutricional = (dif_prote, dif_fibra, dif_grasa, dif_hc, dif_na)

print('Diferencia de proteina :', dif_prote,'gramos')
print('Diferencia de fibra :', dif_fibra, 'gramos')
print('Diferencia de grasas :', dif_grasa, 'gramos')
print('Diferencia de carbohidratos :', dif_hc, 'gramos')
print('Diferencia de sodio :', dif_na, 'gramos')

Viendo los resultados notamos que los nutrientes mas afectados son las proteinas y las grasas. Por lo que buscamos alimentos que tengan una buena relacion precio-proteina y precio-grasas.

Una vez analizada la tabla de precios y el valor nutricional de cada alimento, decidimos remplazar a la carne por harina de trigo y aceite de girasol. Y coincide con las proyecciones en el espacio generado PCA (están cerca).

In [ ]:
tabla_nutricional

In [ ]:
# Corrección del ejercicio
cant_harina = tabla_nutricional[tabla_nutricional['Alimento'] == 'Harina trigo']['Cantidad (gr/ml)']
prote_harina = tabla_nutricional[tabla_nutricional['Alimento'] == 'Harina trigo']['Proteinas (gr)']
harina_compra = (dif_prote * cant_harina)/prote_harina

cant_aceite = tabla_nutricional[tabla_nutricional['Alimento'] == 'Aceite girasol']['Cantidad (gr/ml)']
grasa_aceite = tabla_nutricional[tabla_nutricional['Alimento'] == 'Aceite girasol']['Grasas (gr)']
aceite_compra = (dif_grasa * cant_aceite)/grasa_aceite

total_harina = (harina_compra.reset_index(drop = True)[0] * consumidores_libres[consumidores_libres['PRODUCTOS'] == 'HARINA DE TRIGO']['30/4/2024']) / consumidores_libres[consumidores_libres['PRODUCTOS'] == 'HARINA DE TRIGO']['Cantidad']

total_aceite = (aceite_compra.reset_index(drop = True)[0] * consumidores_libres[consumidores_libres['PRODUCTOS'] == 'ACEITE COCINERO GIRASOL']['30/4/2024']) / consumidores_libres[consumidores_libres['PRODUCTOS'] == 'ACEITE COCINERO GIRASOL']['Cantidad']

print('El valor total de la harina y el aceite necesarios es de', total_harina.reset_index(drop = True)[0] + total_aceite.reset_index(drop = True)[0],'$')

In [ ]:
#Calculo la cantidad de harina y aceite necesaria para remplazar a la carne. Y veo los precios. 

#Multiplico la dif_prote por el costo por gramo de proteína de la harina de trigo en el ultimo mes (Agarramos el precio mas alto).
valor_prote_harina = dif_prote * precio_ev['Proteinas (gr)'][84]

#Multiplico la dif_grasa por el costo por gramo de grasa del aceite cocinero girasol en el ultimo mes (Agarramos el precio mas alto).
valor_grasa_aceite = dif_grasa * precio_ev['Grasas (gr)'][80]

print('El valor total de la harina y el aceite necesarios es de',valor_prote_harina + valor_grasa_aceite,'$')

Estariamos comprando 76.41 gramos de Harina de trigo y 2.96 gramos/mililitros de Aceite de girasol.

In [ ]:
#Este es el valor maximo que tengo para comprar
precio_max = (prom_carnes_ord['Precio'][5] - prom_carnes_ord['Precio'][1])/2

#Por lo que nos estarian sobrando 963$ aproximadamente.

print('Dinero que nos sobra:', precio_max - (valor_grasa_aceite + valor_prote_harina), '$')

#### Consigna 8

Despues de analizar el dominio generado por ACP decidimos elegir a el queso de rallar y la leche en polvo como los dos alimentos para recuperar la proteina perdida de la carne. Hicimos esta eleccion porque notamos que estos dos alimentos se encotraban lejos (en sentido de distancia Euclideana) de las carnes y aparte tenian una gran cantidad de proteinas.

In [ ]:
proteinas_a_recuperar = dif_prote

#Calculo la cantidad de queso rallado y leche en polvo necesaria para recuprar la proteina perdida 
#(estamos recuperando la mitad de proteina con cada alimento).

cantidad_leche_polvo = (proteinas_a_recuperar/2) * (tn[tn['Alimento'] == 'Leche entera en polvo']['Cantidad (gr/ml)'].values[0] / tn[tn['Alimento'] == 'Leche entera en polvo']['Proteinas (gr)']).values[0] #(Cantidad/proteinas) asi veo cuantos gramos de queso de rallar necesito para tener 1 gr de proteina.

cantidad_queso = (proteinas_a_recuperar/2) * (tn[tn['Alimento'] == 'Queso rallar']['Cantidad (gr/ml)'].values[0] / tn[tn['Alimento'] == 'Queso rallar']['Proteinas (gr)']).values[0] #(Cantidad/proteinas) asi veo cuantos gramos de leche en polvo necesito para tener 1 gr de proteina.

#Veo cuanto aumentamos la cantidad de queso rallar y leche en polvo en porcentaje.

aumento_queso = cantidad_queso/ tn[tn['Alimento'] == 'Queso rallar']['Cantidad (gr/ml)'].values[0]
aumento_leche = cantidad_leche_polvo / tn[tn['Alimento'] == 'Leche entera en polvo']['Cantidad (gr/ml)'].values[0]

#Creo una tabla con la cantidad de queso de rallar y leche en polvo que agregamos
tab_queso_leche = tn.copy()

queso_actualizado = tab_queso_leche.iloc[5,1:]
queso_actualizado *= aumento_queso
tab_queso_leche.iloc[5, 1:] = queso_actualizado

leche_actualizada = tab_queso_leche.iloc[1, 1:]
leche_actualizada *= aumento_leche
tab_queso_leche.iloc[1, 1:] = leche_actualizada
tab_queso_leche.head(6)

In [ ]:
#Vemos si ahora pasa las metas de la OMS

print('Cumple las metas de la OMS?', dieta_saludable(tab_queso_leche))

Aún con todos estos cambios la dieta de la canasta basica no pasa las metas que propone la OMS.